## 8.1 Installation & imports

In [24]:
#!pip install -q xgboost mlflow scikit-learn matplotlib pandas
#!pip install deep-translator

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, learning_curve, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

# MLflow est OPTIONNEL : le code fonctionne même s'il n'est pas installé
try:
    import mlflow, mlflow.sklearn
    MLFLOW_DISPONIBLE = True
    print("✅ MLflow disponible :", mlflow.__version__)
except ImportError:
    MLFLOW_DISPONIBLE = False
    print("ℹ️ MLflow non installé — l'entraînement se fera sans tracking.")

print("✅ Imports OK")

In [40]:
df = pd.read_csv("data/raw_collected.csv")

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_10696\1385599139.py:1: DtypeWarning: Columns (11,12,13,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw_collected.csv")


In [4]:
df2 = pd.read_csv("data/sample_reviews2.csv")
df2

,id,source,texte,produit,note,date,auteur_pseudo
0,1,amazon,"Ce produit est absolument fantastique, je reco...",Casque Bluetooth,5,2024-01-15,user_42
1,2,amazon,"Très déçu, le casque est tombé en panne après ...",Casque Bluetooth,1,2024-01-20,user_71
2,3,amazon,"Bon rapport qualité-prix, livraison rapide.",Casque Bluetooth,4,2024-01-22,user_13
3,4,amazon,"Qualité médiocre, ne vaut pas son prix.",Casque Bluetooth,2,2024-02-01,user_58
4,5,amazon,Parfait ! Son excellent et confortable à porter.,Casque Bluetooth,5,2024-02-05,user_99
5,6,fnac,"Produit conforme à la description, rien à redire.",Montre Connectée,3,2024-02-10,user_27
6,7,fnac,"Incroyable montre, toutes les fonctions marche...",Montre Connectée,5,2024-02-14,user_64
7,8,fnac,"La batterie ne tient pas une journée, très déc...",Montre Connectée,1,2024-02-18,user_31
8,9,fnac,"Bonne montre pour le prix, quelques bugs mineurs.",Montre Connectée,3,2024-02-25,user_87
9,10,fnac,"Service après-vente excellent, problème résolu...",Montre Connectée,4,2024-03-01,user_52


In [5]:
df3 = pd.read_csv("data/sample_reviews.csv")
df3

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [7]:
df3['id_2'] = df3['Id'] + 25


In [9]:
df3['source'] = 'amazon'

In [13]:
import time
df3['date'] = pd.to_datetime(df3['Time']).dt.strftime('%Y-%m-%d')

In [14]:
df3

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,id_2,source,date
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,26,amazon,1970-01-01
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,27,amazon,1970-01-01
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,28,amazon,1970-01-01
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,29,amazon,1970-01-01
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,30,amazon,1970-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...,568475,amazon,1970-01-01
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...,568476,amazon,1970-01-01
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o...",568477,amazon,1970-01-01
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...,568478,amazon,1970-01-01


In [28]:
df2.columns

Index(['id', 'source', 'texte', 'produit', 'note', 'date', 'auteur_pseudo'], dtype='object')

In [47]:
df3.columns

Index(['ProductId', 'UserId', 'Score', 'Text', 'id_2', 'source', 'date'], dtype='object')

In [49]:
df3.columns = ['produit', 'auteur_pseudo', 'note', 'texte', 'id', 'source', 'date']

In [50]:
df3

,produit,auteur_pseudo,note,texte,id,source,date
0,B001E4KFG0,A3SGXH7AUHU8GW,5,I have bought several of the Vitality canned d...,26,amazon,1970-01-01
1,B00813GRG4,A1D87F6ZCVE5NK,1,Product arrived labeled as Jumbo Salted Peanut...,27,amazon,1970-01-01
2,B000LQOCH0,ABXLMWJIXXAIN,4,This is a confection that has been around a fe...,28,amazon,1970-01-01
3,B000UA0QIQ,A395BORC6FGVXV,2,If you are looking for the secret ingredient i...,29,amazon,1970-01-01
4,B006K2ZZ7K,A1UQRSCLF8GW1T,5,Great taffy at a great price. There was a wid...,30,amazon,1970-01-01
...,...,...,...,...,...,...,...
568449,B001EO7N10,A28KG5XORO54AY,5,Great for sesame chicken..this is a good if no...,568475,amazon,1970-01-01
568450,B003S1WTCU,A3I8AFVPEE8KI5,2,I'm disappointed with the flavor. The chocolat...,568476,amazon,1970-01-01
568451,B004I613EE,A121AA1GQV751Z,5,"These stars are small, so you can give 10-15 o...",568477,amazon,1970-01-01
568452,B004I613EE,A3IBEVCTXKNOH,5,These are the BEST treats for training and rew...,568478,amazon,1970-01-01


In [ ]:
# Nettoyage : on retire les colonnes d'origine inutiles SI elles existent encore.
# errors="ignore" rend la cellule ré-exécutable même après le renommage ci-dessus
# (sinon KeyError quand on relance le notebook de haut en bas).
df3 = df3.drop(
    columns=['Id', 'ProfileName', 'HelpfulnessNumerator',
             'HelpfulnessDenominator', 'Time', 'Summary', 'Text'],
    errors="ignore",
)
df3

In [ ]:
# On garde la trace de la langue d'origine pour ne traduire que l'anglais
df2["langue"] = "fr"   # avis déjà rédigés en français
df3["langue"] = "en"   # avis Amazon en anglais (seront traduits)
df_final = pd.concat([df2, df3], ignore_index=True)

In [ ]:
df_final

In [41]:
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,id,source,texte,produit,note,date,auteur_pseudo
0,1.0,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1.0,1.0,5.0,1.303862e+09,Good Quality Dog Food,I have bought several of the Vitality canned d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0.0,0.0,1.0,1.346976e+09,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1.0,1.0,4.0,1.219018e+09,"""Delight"" says it all",This is a confection that has been around a fe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,B000UA0QIQ,A395BORC6FGVXV,Karl,3.0,3.0,2.0,1.307923e+09,Cough Medicine,If you are looking for the secret ingredient i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0.0,0.0,5.0,1.350778e+09,Great taffy,Great taffy at a great price. There was a wid...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2005.0,scraping,Try not to become a man of success. Rather bec...,Produit Scraping,3.0,2026-06-19,albert_einstein
568480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006.0,scraping,It is better to be hated for what you are than...,Produit Scraping,3.0,2026-06-19,andré_gide
568481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.0,scraping,"I have not failed. I've just found 10,000 ways...",Produit Scraping,3.0,2026-06-19,thomas_a._edison
568482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.0,scraping,A woman is like a tea bag; you never know how ...,Produit Scraping,3.0,2026-06-19,eleanor_roosevelt


## 8.2 Récupération des données

In [ ]:
# === 8.2 Préparation des données à partir de df_final ===
# 1) label binaire à partir de la note  2) sous-échantillon équilibré
# 3) traduction des avis anglais en français (avec cache)  4) split train/val
import os, re, time

CACHE_FR     = "data/df_final_fr.csv"   # cache de la traduction (évite de retraduire)
N_PAR_CLASSE = 1000                      # ~2000 lignes équilibrées au total
RANDOM_STATE = 42

def _nettoyer(txt):
    txt = re.sub(r"<[^>]+>", " ", str(txt))    # retire le HTML (<br />, ...)
    return re.sub(r"\s+", " ", txt).strip()

if os.path.exists(CACHE_FR):
    # --- on repart du cache déjà traduit ---
    ech = pd.read_csv(CACHE_FR)
    print(f"✅ Données françaises chargées depuis le cache : {CACHE_FR} ({len(ech)} lignes)")
else:
    # 1) Label binaire : note >= 4 -> 1 (POSITIVE), note <= 2 -> 0 (NEGATIVE), on retire la note 3
    d = df_final.copy()
    d["note"]  = pd.to_numeric(d["note"], errors="coerce")
    d          = d[d["note"] != 3].dropna(subset=["note", "texte"])
    d["label"] = (d["note"] >= 4).astype(int)
    d["texte"] = d["texte"].map(_nettoyer)
    d          = d[d["texte"].str.len() > 0]
    if "langue" not in d.columns:
        d["langue"] = "en"
    print(f"Après étiquetage : {len(d)} lignes | répartition {d['label'].value_counts().to_dict()}")

    # 2) Sous-échantillon équilibré (~2000 lignes)
    pos = d[d.label == 1].sample(n=min(N_PAR_CLASSE, int((d.label == 1).sum())), random_state=RANDOM_STATE)
    neg = d[d.label == 0].sample(n=min(N_PAR_CLASSE, int((d.label == 0).sum())), random_state=RANDOM_STATE)
    ech = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Échantillon équilibré : {len(ech)} lignes | {ech['label'].value_counts().to_dict()}")

    # 3) Traduction anglais -> français (par lots, robuste aux erreurs)
    def _traduire_en_fr(textes):
        from deep_translator import GoogleTranslator
        tr, out = GoogleTranslator(source="en", target="fr"), []
        for i in range(0, len(textes), 50):
            lot = [t[:4900] for t in textes[i:i+50]]   # limite Google ~5000 caractères
            try:
                out.extend(tr.translate_batch(lot))
            except Exception as e:
                print(f"  ⚠️ lot {i}-{i+len(lot)} non traduit ({e}) — texte original conservé")
                out.extend(lot)
            time.sleep(0.3)
        return out

    masque_en = ech["langue"].eq("en")
    print(f"🌐 Traduction de {int(masque_en.sum())} avis anglais en français (peut prendre quelques minutes)...")
    ech["texte_fr"] = ech["texte"]
    ech.loc[masque_en, "texte_fr"] = _traduire_en_fr(ech.loc[masque_en, "texte"].tolist())
    ech.to_csv(CACHE_FR, index=False)
    print(f"✅ Traduction terminée et mise en cache : {CACHE_FR}")

# 4) Découpage train / validation (stratifié) sur les textes FRANÇAIS
X = ech["texte_fr"].astype(str).tolist()
y = ech["label"].astype(int).tolist()
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
y_train = list(map(int, y_train)); y_val = list(map(int, y_val))
print(f"   Train : {len(X_train)} exemples | Val : {len(X_val)} exemples | positifs val : {sum(y_val)}")

## 8.3 Le pipeline XGBoost

`TfidfVectorizer` transforme le texte en vecteurs numériques, puis `XGBClassifier`
(gradient boosting d'arbres de décision) effectue la classification binaire.

In [27]:
def build_pipeline(max_features=20_000, ngram_range=(1, 2), min_df=1,
                   n_estimators=300, max_depth=6, learning_rate=0.1,
                   subsample=0.9, colsample_bytree=0.9, random_state=42):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=max_features, ngram_range=ngram_range,
            min_df=min_df, sublinear_tf=True,
        )),
        ("clf", XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, subsample=subsample,
            colsample_bytree=colsample_bytree,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=random_state,
        )),
    ])

print("✅ build_pipeline() prêt")
build_pipeline()

✅ build_pipeline() prêt


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.9, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=-1,
                               num_parallel_tree=None, ...))])

## 8.4 GridSearch — recherche d'hyperparamètres

`GridSearchCV` teste toutes les combinaisons d'hyperparamètres en validation croisée
et conserve la meilleure selon le score F1.

In [ ]:
param_grid = {
    "clf__n_estimators":  [200, 400],
    "clf__max_depth":     [4, 6],
    "clf__learning_rate": [0.1, 0.3],
}

print("🔎 Lancement du GridSearchCV (cela peut prendre quelques minutes)...")
t0 = time.time()
grid = GridSearchCV(
    estimator=build_pipeline(),
    param_grid=param_grid,
    cv=3, scoring="f1", n_jobs=-1, verbose=1, refit=True,
)
grid.fit(X_train, y_train)
temps_train_xgb = time.time() - t0

best_xgb = grid.best_estimator_
print(f"\n✅ GridSearch terminé en {temps_train_xgb:.1f}s")
print(f"   Meilleurs hyperparamètres : {grid.best_params_}")
print(f"   Meilleur F1 (CV)          : {grid.best_score_:.4f}")

## 8.5 Métriques de classification

Évaluation du meilleur modèle sur le jeu de validation :
accuracy, precision, recall, F1, ROC-AUC, matrice de confusion et rapport détaillé.

In [ ]:
def evaluate_classification(model, X_test, y_test, target_names=("NEGATIVE", "POSITIVE")):
    y_pred = model.predict(X_test)
    etiquettes = list(range(len(target_names)))  # [0, 1] -> forme binaire garantie
    roc_auc = None
    if hasattr(model, "predict_proba"):
        try:
            roc_auc = float(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
        except Exception:
            roc_auc = None
    return {
        "accuracy":  float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_test, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_test, y_pred, zero_division=0)),
        "roc_auc":   roc_auc,
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=etiquettes).tolist(),
        "classification_report": classification_report(
            y_test, y_pred, labels=etiquettes, target_names=list(target_names), zero_division=0),
    }

t0 = time.time()
metrics_xgb = evaluate_classification(best_xgb, X_val, y_val)
temps_inf_xgb = (time.time() - t0) / len(X_val) * 1000  # ms / exemple

print("📊 XGBoost — métriques de validation")
print(f"   Accuracy  : {metrics_xgb['accuracy']:.4f} ({metrics_xgb['accuracy']*100:.2f}%)")
print(f"   Precision : {metrics_xgb['precision']:.4f}")
print(f"   Recall    : {metrics_xgb['recall']:.4f}")
print(f"   F1-Score  : {metrics_xgb['f1']:.4f}")
print(f"   ROC-AUC   : {metrics_xgb['roc_auc']:.4f}" if metrics_xgb['roc_auc'] else "   ROC-AUC   : n/a")
print(f"   Latence   : {temps_inf_xgb:.3f} ms/exemple")
print("\n" + metrics_xgb["classification_report"])

### 8.5.1 Matrice de confusion

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    np.array(metrics_xgb["confusion_matrix"]),
    display_labels=["NEGATIVE", "POSITIVE"],
).plot(ax=ax, colorbar=False, cmap="Greens")
ax.set_title("Matrice de confusion — XGBoost", fontweight="bold")
plt.tight_layout()
plt.show()

## 8.6 Courbe d'apprentissage (Learning Curve)

La courbe d'apprentissage trace la performance en fonction de la taille du jeu
d'entraînement. Elle aide à diagnostiquer le **sur-apprentissage** (écart train/val élevé)
et le **sous-apprentissage** (scores bas qui plafonnent).

In [ ]:
print("📈 Calcul de la courbe d'apprentissage (validation croisée)...")
sizes, train_scores, val_scores = learning_curve(
    estimator=build_pipeline(n_estimators=200, max_depth=4),
    X=list(X_train), y=list(y_train),
    cv=3, scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1, shuffle=True, random_state=42,
)
train_mean, train_std = train_scores.mean(1), train_scores.std(1)
val_mean,   val_std   = val_scores.mean(1),   val_scores.std(1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sizes, train_mean, "o-", color="#6366F1", label="Score d'entraînement")
ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#6366F1")
ax.plot(sizes, val_mean, "o-", color="#F97316", label="Score de validation (CV)")
ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#F97316")
ax.set_title("Courbe d'apprentissage — XGBoost", fontweight="bold")
ax.set_xlabel("Nombre d'exemples d'entraînement")
ax.set_ylabel("Score (F1)")
ax.legend(loc="best"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8.7 Suivi de l'expérience avec MLflow

On enregistre dans MLflow : les **hyperparamètres** retenus, les **métriques** de validation,
le **rapport de classification** (artefact) et le **modèle** sérialisé.

> Pour visualiser ensuite : lancer `mlflow ui` dans un terminal à la racine du projet,
> puis ouvrir http://127.0.0.1:5000

In [ ]:
if MLFLOW_DISPONIBLE:
    mlflow.set_experiment("sentiment-xgboost")
    with mlflow.start_run(run_name="xgboost-gridsearch") as run:
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("cv_folds", 3)
        mlflow.log_param("scoring", "f1")
        mlflow.log_param("algo", "XGBClassifier")
        for cle in ("accuracy", "precision", "recall", "f1", "roc_auc"):
            if metrics_xgb.get(cle) is not None:
                mlflow.log_metric(cle, metrics_xgb[cle])
        # on évite de logguer un score nan (sinon erreur SQLite)
        if grid.best_score_ == grid.best_score_:  # False si nan
            mlflow.log_metric("cv_best_f1", grid.best_score_)
        mlflow.log_text(metrics_xgb["classification_report"], "classification_report.txt")
        try:
            mlflow.sklearn.log_model(best_xgb, name="model")
        except Exception as e:
            print("⚠️ log_model ignoré :", e)
        print(f"✅ Run MLflow enregistré : {run.info.run_id}")
        print("   → lancer `mlflow ui` puis ouvrir http://127.0.0.1:5000")
else:
    print("ℹ️ MLflow non installé : `pip install mlflow` pour activer le tracking.")

## 8.8 Ajout de XGBoost au tableau comparatif

Si la section 6 (tableau `df_resultats`) a déjà été exécutée, on y ajoute la ligne XGBoost.

In [ ]:
ligne_xgb = {
    "Modèle": "XGBoost (GridSearch)",
    "Approche": "From Scratch",
    "Accuracy": metrics_xgb["accuracy"],
    "F1-Score": metrics_xgb["f1"],
    "Latence (ms/ex)": temps_inf_xgb,
    "Taille modèle": "~5-15 MB",
    "Temps entraîn.": f"{temps_train_xgb:.1f}s (GridSearch)",
}

try:
    df_resultats = pd.concat([df_resultats, pd.DataFrame([ligne_xgb])], ignore_index=True)  # noqa: F821
    df_resultats["Accuracy (%)"] = (df_resultats["Accuracy"] * 100).round(2)
    df_resultats["F1 (%)"] = (df_resultats["F1-Score"] * 100).round(2)
    print(df_resultats[["Modèle", "Approche", "Accuracy (%)", "F1 (%)",
                        "Latence (ms/ex)", "Taille modèle", "Temps entraîn."]].to_string(index=False))
except NameError:
    print("Tableau comparatif (df_resultats) non trouvé — résultats XGBoost seuls :")
    print(pd.DataFrame([ligne_xgb]).to_string(index=False))

## 8.9 Tests automatisés (C12)

La logique XGBoost est packagée dans **`ml/xgboost_sentiment.py`** et couverte par
des tests **pytest** dans **`tests/test_xgboost_sentiment.py`** (pipeline, métriques,
GridSearch, learning curve, entraînement + MLflow).

Pour les lancer depuis un terminal à la racine du projet :

```bash
pytest tests/test_xgboost_sentiment.py -v
```

On peut aussi exécuter cette cellule pour les lancer directement depuis le notebook :

In [34]:
import subprocess, sys
res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_xgboost_sentiment.py", "-v"],
    capture_output=True, text=True,
)
print(res.stdout[-3000:])
print(res.stderr[-1000:])

============================= test session starts =============================
platform win32 -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0 -- C:\Users\Utilisateur\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Utilisateur\Documents\certif\avis-sentiment
plugins: anyio-4.2.0
collecting ... collected 7 items

tests/test_xgboost_sentiment.py::test_build_pipeline_structure PASSED    [ 14%]
tests/test_xgboost_sentiment.py::test_pipeline_fit_predict PASSED        [ 28%]
tests/test_xgboost_sentiment.py::test_evaluate_classification_keys PASSED [ 42%]
tests/test_xgboost_sentiment.py::test_evaluate_classification_valeurs PASSED [ 57%]
tests/test_xgboost_sentiment.py::test_run_gridsearch PASSED              [ 71%]
tests/test_xgboost_sentiment.py::test_compute_learning_curve PASSED      [ 85%]
tests/test_xgboost_sentiment.py::test_train_with_mlflow PASSED           [100%]

============================= 7 passed in 26.11s ==============================


